# Finalize: all results in one file (2004-2026.09)

**notebook version**: `1` (26 Oct 2026)

## ℹ️ About this notebook

This notebook collects every result available so far into one file, on one 30-minute timestamp
from the start of the meteo record in 2004 to 30 September 2026. The datasets of a release are cut
from it by period, for example FP2026.1 (2016 to September 2026) or a single setup period such as
`2026_1`:

```python
finalize.loc['2026-01-01 00:15':'2026-09-30 23:45']
```

| Source | What it adds | Period |
|---|---|---|
| `10_METEO/30_PRODUCTS/99_METEO_MERGED_2004-2026.09` | meteo products `01` to `07` (`SW_IN`, `TA`, `PPFD_IN`, `RH`, `PA`, `LW_IN`, `VPD`) | 2004 to September 2026 |
| `10_METEO/30_PRODUCTS/08` to `13` | `PREC`, `SWC`, `TS`, `G`, `SW_OUT`, `LW_OUT` | 2004 to 2025; not yet extended to 2026 |
| `30_FLUX_PROCESSING_CHAIN/34_IRGA72_2016-2024/21_FPPC_IRGA72_MERGED_2016-2026.09` | `IRGA72` Level-1 fluxes and the results of the `NEE`, `LE` and `H` chains | 2016 to September 2026 |

Each source adds only the columns the file does not have yet. A column that two sources share must
hold the same values in both. The fluxes of the open-path LI-7500 (`IRGA75`, 2004 to 2017 and 2019)
are not part of this file yet, because their chain has not been run.

## ⚙️ Settings

In [1]:
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
PRODUCTS = rf"{DATADIR}\10_METEO\30_PRODUCTS"
# Each meteo product at its latest period; 08 to 13 still end in 2025.
FILES = {
    'meteo 01-07': rf"{PRODUCTS}\99_METEO_MERGED_2004-2026.09.parquet",
    'meteo 08 PREC': rf"{PRODUCTS}\08_METEO_PREC_GAPFILLED_2004-2025.parquet",
    'meteo 09 SWC': rf"{PRODUCTS}\09_METEO_SWC_FF1_2004-2025.parquet",
    'meteo 10 TS': rf"{PRODUCTS}\10_METEO_TS_FF1_2004-2025.parquet",
    'meteo 11 G': rf"{PRODUCTS}\11_METEO_G_FF1_2004-2025.parquet",
    'meteo 12 SW_OUT': rf"{PRODUCTS}\12_METEO_SW_OUT_2005-2025.parquet",
    'meteo 13 LW_OUT': rf"{PRODUCTS}\13_METEO_LW_OUT_2005-2025.parquet",
    'fluxes IRGA72': (rf"{DATADIR}\30_FLUX_PROCESSING_CHAIN\34_IRGA72_2016-2024"
                      r"\21_FPPC_IRGA72_MERGED_2016-2026.09.parquet"),
}
PERIOD_START, PERIOD_END = '2004-01-01 00:15', '2026-09-30 23:45'

OUTNAME = "81_FINALIZE_2004-2026.09"  # same name as this notebook
OUTPATH = rf"{DATADIR}\80_FINALIZE"

In [2]:
from datetime import datetime
import importlib.metadata

import diive as dv
import numpy as np
import pandas as pd

NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Last run: 2026-10-26 16:16:44
diive version: v0.93.0


## 📂 Load data

In [3]:
frames = {k: dv.load_parquet(filepath=v) for k, v in FILES.items()}
for name, frame in frames.items():
    assert frame.index.is_unique, f"{name} has duplicate timestamps"
    print(f"{name:>16}: {frame.shape[0]:,} records x {frame.shape[1]:>3} columns, "
          f"{frame.index[0]} -> {frame.index[-1]}")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\99_METEO_MERGED_2004-2026.09.parquet (0.061 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\08_METEO_PREC_GAPFILLED_2004-2025.parquet (0.011 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\09_METEO_SWC_FF1_2004-2025.parquet (0.024 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\10_METEO_TS_FF1_2004-2025.parquet (0.036 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\11_METEO_G_FF1_2004-2025.parquet (0.016 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\12_METEO_SW_OUT_2005-2025.parquet (0.012 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\13_METEO_LW_OUT_2005-2025.parquet (0.015 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_PROCESSIN
G_CHAIN\34_IRGA72_2016-2024\21_FPPC_IRGA72_MERGED_2016-2026.09.parquet (0.330 seconds).

     meteo 01-07: 398,832 records x  19 columns, 2004-01-01 00:15:00 -> 2026-09-30 23:45:00
   meteo 08 PREC: 385,728 records x   4 columns, 2004-01-01 00:15:00 -> 2025-12-31 23:45:00
    meteo 09 SWC: 373,728 records x  14 columns, 2004-09-07 00:15:00 -> 2025-12-31 23:45:00
     meteo 10 TS: 373,728 records x  35 columns, 2004-09-07 00:15:00 -> 2025-12-31 23:45:00
      meteo 11 G: 373,728 records x   8 columns, 2004-09-07 00:15:00 -> 2025-12-31 23:45:00
 meteo 12 SW_OUT: 355,872 records x   3 columns, 2005-09-14 00:15:00 -> 2025-12-31 23:45:00
 meteo 13 LW_OUT: 355,872 records x   2 columns, 2005-09-14 00:15:00 -> 2025-12-31 23:45:00
   fluxes IRGA72: 188,448 records x 487 columns, 2016-01-01 00:15:00 -> 2026-09-30 23:45:00


## Collect

In [4]:
def add_new_columns(main, frame, name, source):
    """Add the columns of frame that main does not have yet, on main's timestamps.

    A column that main already has must hold the same values in frame, so that no result is
    dropped silently. Returns the extended frame and records each new column's source.
    """
    assert frame.index.isin(main.index).all(), f"{name} has timestamps outside the main index"
    frame = frame.reindex(main.index)
    new = [c for c in frame.columns if c not in main.columns]
    for c in frame.columns.difference(new):
        a, b = main[c], frame[c]
        both = a.notna() & b.notna()
        assert np.allclose(a[both], b[both], equal_nan=True), f"{c} differs between main and {name}"
    for c in new:
        source[c] = name
    print(f"{name}: {len(new)} new columns, {frame.shape[1] - len(new)} already present")
    return pd.concat([main, frame[new]], axis=1)

In [5]:
index = pd.date_range(PERIOD_START, PERIOD_END, freq='30min', name='TIMESTAMP_MIDDLE')
finalize = pd.DataFrame(index=index)
SOURCE = {}
for name, frame in frames.items():
    finalize = add_new_columns(finalize, frame, name, SOURCE)
assert finalize.columns.is_unique
fullset = finalize  # name used by the export cell
print(f"Finalize: {finalize.shape[0]:,} records x {finalize.shape[1]} columns, "
      f"{finalize.memory_usage().sum() / 1e9:.1f} GB in memory")

meteo 01-07: 19 new columns, 0 already present
meteo 08 PREC: 4 new columns, 0 already present
meteo 09 SWC: 14 new columns, 0 already present
meteo 10 TS: 35 new columns, 0 already present


meteo 11 G: 8 new columns, 0 already present
meteo 12 SW_OUT: 3 new columns, 0 already present
meteo 13 LW_OUT: 2 new columns, 0 already present


fluxes IRGA72: 487 new columns, 0 already present
Finalize: 398,832 records x 572 columns, 1.8 GB in memory


### 📋 Table: columns per source

In [6]:
display(pd.Series(SOURCE).value_counts().reindex(list(FILES)).rename('columns').to_frame())

,columns
meteo 01-07,19
meteo 08 PREC,4
meteo 09 SWC,14
meteo 10 TS,35
meteo 11 G,8
meteo 12 SW_OUT,3
meteo 13 LW_OUT,2
fluxes IRGA72,487


### 📋 Table: coverage per source and year

The share of records per year in which a source has at least one value. A source that ends before
30 September 2026 shows 0 in the years after its end.

In [7]:
_cov = {}
for _name in FILES:
    _cols = [c for c, s in SOURCE.items() if s == _name]
    _cov[_name] = finalize[_cols].notna().any(axis=1).groupby(finalize.index.year).mean() * 100
display(pd.DataFrame(_cov).round(0).astype(int).rename_axis('year'))

,meteo 01-07,meteo 08 PREC,meteo 09 SWC,meteo 10 TS,meteo 11 G,meteo 12 SW_OUT,meteo 13 LW_OUT,fluxes IRGA72
year,,,,,,,,
2004,100,100,32,32,32,0,0,0
2005,100,100,100,100,100,30,30,0
2006,100,100,100,100,100,100,100,0
2007,100,100,100,100,100,100,100,0
2008,100,100,100,100,100,100,100,0
2009,100,100,100,100,100,100,100,0
2010,100,100,100,100,100,100,100,0
2011,100,100,100,100,100,100,100,0
2012,100,100,100,100,100,100,100,0


## 💾 Export

The final set as parquet, and next to it an `_INFO.md` file that lists every column with its source.

In [8]:
filepath = dv.save_parquet(data=finalize, filename=OUTNAME, outpath=OUTPATH)

_lines = [f"# {OUTNAME}", "",
          f"{fullset.index[0]} to {fullset.index[-1]} ({fullset.index.name}, UTC+01:00), "
          f"{fullset.shape[0]:,} records x {fullset.shape[1]} columns.",
          f"Written {datetime.now():%Y-%m-%d %H:%M} by `{OUTNAME}.ipynb`.", "",
          "| source | file |", "|---|---|"]
_lines += [f"| {k} | `{v}` |" for k, v in FILES.items()]
for _name in FILES:
    _cols = [c for c, s in SOURCE.items() if s == _name]
    _lines += ["", f"## {_name} ({len(_cols)} columns)", ""] + [f"- `{c}`" for c in _cols]
with open(rf"{OUTPATH}\{OUTNAME}_INFO.md", "w", encoding="utf-8") as _f:
    _f.write("\n".join(_lines) + "\n")
print(f"Wrote {OUTPATH}\\{OUTNAME}_INFO.md")

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\81_FI
NALIZE_2004-2026.09.parquet (5.269 seconds).

Wrote F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\80_FINALIZE\81_FINALIZE_2004-2026.09_INFO.md


***

# End of notebook.

In [9]:
NOTEBOOK_END = datetime.now()
print(f"Runtime: {NOTEBOOK_END - NOTEBOOK_START}")

Runtime: 0:00:07.624975
